[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/carlospi2000/Notes_on_Computational_Physics/blob/main/08.Zeros_maxima_minima_function/08_Maximos_Ceros_ESTUDIANTES-4.ipynb)

## Taller 8: la radiación de cuerpo negro
## Cristian Fernando Castro Torres


### A. Enunciado y requisitos

La radiancia espectral de un cuerpo negro a temperatura $T$ está dada por la **ley de Planck**:

$$B_\lambda(\lambda, T) = \frac{2hc^2}{\lambda^5}\,\frac{1}{e^{hc/(\lambda k_B T)} - 1} ,$$

donde $B_\lambda$ es la potencia emitida por unidad de área, de ángulo sólido y de longitud de onda [W m⁻² sr⁻¹ m⁻¹], $\lambda$ la longitud de onda [m], $T$ la temperatura [K], $h = 6.62607015\times10^{-34}$ J s la constante de Planck, $c = 299792458$ m/s la rapidez de la luz en el vacío y $k_B = 1.380649\times10^{-23}$ J/K la constante de Boltzmann. El factor $2hc^2/\lambda^5$ cuenta los modos electromagnéticos con su energía y el factor con el exponencial es la ocupación de esos modos. Estas tres constantes son exactas en el Sistema Internacional.

**Variable adimensional.** Con $x = hc/(\lambda k_B T)$, la razón entre la energía de un fotón y la energía térmica, el espectro queda

$$B_\lambda \propto \frac{x^5}{e^x - 1} .$$

**Ley de desplazamiento de Wien.** El máximo del espectro cumple $dB_\lambda/dx = 0$, que se reduce a

$$x = n\left(1 - e^{-x}\right), \qquad n = 5 ,$$

ecuación trascendente sin solución en funciones elementales, con la raíz trivial $x = 0$ y una raíz positiva $x_5$. Esa raíz fija la constante de Wien $b$:

$$\lambda_{\max}\,T = b = \frac{hc}{k_B\,x_5} .$$

Si el espectro se expresa por unidad de frecuencia, $B_\nu \propto x^3/(e^x - 1)$, y el máximo cumple la misma ecuación con $n = 3$.

**Ley de Stefan-Boltzmann.** La potencia total emitida por unidad de área es $M = \int_0^\infty \pi B_\lambda\,d\lambda = \sigma T^4$, con

$$\sigma = \frac{2\pi k_B^4}{h^3 c^2}\,I, \qquad I = \int_0^\infty \frac{x^3}{e^x - 1}\,dx ,$$

donde $\sigma$ es la constante de Stefan-Boltzmann [W m⁻² K⁻⁴] e $I$ una integral pura, cuyo valor exacto es $\pi^4/15$. Es una integral impropia. El cambio de variable $x = t/(1 - t)$, con $dx = dt/(1-t)^2$, la lleva al intervalo finito $[0, 1]$:

$$I = \int_0^1 \frac{x(t)^3}{e^{x(t)} - 1}\,\frac{dt}{(1 - t)^2} .$$

**El taller tiene tres partes:**

| Parte | Lenguaje | Qué se hace |
|---|---|---|
| B | Fortran | Módulo con tres métodos para hallar ceros; constante de Wien |
| C | Fortran | Módulo de Romberg del Taller 7 con un integrando nuevo; constante de Stefan-Boltzmann |
| D | Python | Máximo del espectro solar en una malla, con spline, comparado con la ley de Wien |

**Requisitos generales**

1. Todo el trabajo, incluido este notebook, se guarda en una carpeta llamada `taller_cuerpo_negro`, con los nombres de archivo que se indican en cada parte. El archivo `modulo_romberg.f90` del Taller 7 se copia en esa carpeta.
2. Se usa doble precisión: `real(dp)` en Fortran y `float` (64 bits) en Python.
3. Los métodos para hallar ceros tienen un máximo de 100 iteraciones y terminan con un mensaje si no convergen. Romberg conserva el límite de niveles del Taller 7.
4. La bisección comprueba el cambio de signo antes de iterar. Newton y secante se detienen con un mensaje si el denominador es nulo.
5. Se compila con `-fcheck=bounds`.
6. Los números reales de los archivos `.dat` se escriben con formato `es24.16`, y los identificadores y conteos como enteros.

**Qué se entrega**

- En la carpeta del repositorio: los programas, los archivos de datos, la figura y este notebook, con la celda de verificación en 5 de 5 PASS y las respuestas de la sección E en celdas Markdown al final.
- En Moodle: el reporte de una página y el enlace a la carpeta.

<div style="border-left: 5px solid #B02A20; background-color: #FBEDEC; padding: 10px 14px; margin: 8px 0;">

**No está permitido** usar `scipy.optimize`, `scipy.integrate`, `np.gradient`, `np.trapz` ni funciones de biblioteca que hallen ceros, integren o deriven. `CubicSpline`, con sus métodos `.roots()` y `.derivative()`, se permite solo en la Parte D; las diferencias centrales se programan. Los métodos se programan a partir de las secciones 2 a 5. Cada estudiante sustentará su programa.

</div>

### B. Parte Fortran: constante de Wien

**Actividades**

1. Escriba `modulo_ceros.f90` con la función $F(x) = x - n(1 - e^{-x})$, su derivada y los métodos de bisección, Newton y secante. La organización en procedimientos es decisión suya; $n$ se pasa como argumento y debe respetar las reglas de la Semana 6.
2. Escriba `programa_wien.f90`, que use ese módulo para hallar $x_5$ y $x_3$ con tolerancia $\varepsilon = 10^{-10}$, con estos datos iniciales fijos:

| Método | Datos iniciales | Parada |
|---|---|---|
| Bisección | $[a, b] = [1, 10]$ | $b - c \le \varepsilon$ (sección 2) |
| Newton | $x_0 = 10$ | $\lvert x_{k+1} - x_k\rvert < \varepsilon$ |
| Secante | $x_0 = 10$, $x_1 = 9$ | $\lvert x_{k+1} - x_k\rvert < \varepsilon$ |

3. Repita Newton para $n = 5$ con $x_0 = 0.5$, y explique el resultado en el reporte.
4. Calcule la constante de Wien $b$ con $x_5$ obtenido por Newton.
5. Registre cada iteración de los tres métodos con $n = 5$ para analizar la convergencia. Las iteraciones de cada método se numeran desde 1 y en orden: la primera es la que produce el primer valor nuevo ($c_1$ en bisección, $x_1$ en Newton, $x_2$ en la secante).

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `modulo_ceros.f90` | Función, derivada y los tres métodos, con conteo de iteraciones |
| `programa_wien.f90` | Usa el módulo; escribe los tres archivos `.dat` de esta parte |
| `resultado_wien.dat` | 7 filas: $n$, método, raíz, iteraciones. Método: 1 bisección, 2 Newton, 3 secante, para $n = 5$ y $n = 3$; la fila 7 es $n = 5$, método 4 (Newton con $x_0 = 0.5$) |
| `iteraciones_wien.dat` | Una fila por iteración con $n = 5$: método (1 a 3), número de iteración, valor de $x$ |
| `constante_wien.dat` | Una línea: $x_5$, $x_3$, $b$ [m K] |

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** La bisección para $n = 5$ debe usar el número de iteraciones que predice la fórmula de la sección 2. Calcúlelo antes de ejecutar el programa: si no coincide, revise el criterio de parada.

</div>

**Solución — Parte B.** Los archivos se escriben con `%%writefile`; el módulo devuelve `estado` (0 convergió, 1 no convergió, 2 sin cambio de signo, 3 denominador nulo) y el historial de iteraciones.

In [19]:
%%writefile modulo_ceros.f90
! ============================================================
! Módulo:    modulo_ceros
! Propósito: F(x) = x - n(1 - e^-x), su derivada y tres métodos
!            para hallar ceros: bisección, Newton y secante.
! Estado:    0 = convergió; 1 = no convergió en max_iter;
!            2 = sin cambio de signo (bisección); 3 = denominador nulo.
! ============================================================
module modulo_ceros
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer, parameter :: max_iter = 100      ! protección contra ciclos sin fin
contains

  ! F: ecuación de Wien, x = n(1 - e^-x), escrita como F(x) = 0
  real(dp) function F(x, n)
    real(dp), intent(in) :: x
    integer,  intent(in) :: n
    F = x - real(n, dp)*(1.0_dp - exp(-x))
  end function F

  ! dF: derivada analítica, F'(x) = 1 - n e^-x
  real(dp) function dF(x, n)
    real(dp), intent(in) :: x
    integer,  intent(in) :: n
    dF = 1.0_dp - real(n, dp)*exp(-x)
  end function dF

  ! Bisección en [a, b]; hist(k) guarda el punto medio c_k
  subroutine biseccion(n, a0, b0, tol, raiz, iter, estado, hist)
    integer,  intent(in)  :: n
    real(dp), intent(in)  :: a0, b0, tol
    real(dp), intent(out) :: raiz, hist(max_iter)
    integer,  intent(out) :: iter, estado
    real(dp) :: a, b, c
    a = a0;  b = b0;  raiz = a0;  iter = 0;  estado = 1;  hist = 0.0_dp
    if (F(a, n)*F(b, n) >= 0.0_dp) then      ! paso 1: cambio de signo
       estado = 2
       print '(a)', 'bisección: F(a) y F(b) no tienen signos distintos'
       return
    end if
    do iter = 1, max_iter
       c = 0.5_dp*(a + b)                    ! paso 2: punto medio
       hist(iter) = c
       if (b - c <= tol) then                ! paso 3: criterio de parada
          raiz = c;  estado = 0;  return
       end if
       if (F(a, n)*F(c, n) <= 0.0_dp) then   ! paso 4: mitad con cambio de signo
          b = c
       else
          a = c
       end if
    end do
    iter = max_iter
    raiz = c
    print '(a)', 'bisección: no convergió en max_iter iteraciones'
  end subroutine biseccion

  ! Newton-Raphson desde x0; hist(k) guarda x_k
  subroutine newton(n, x0, tol, raiz, iter, estado, hist)
    integer,  intent(in)  :: n
    real(dp), intent(in)  :: x0, tol
    real(dp), intent(out) :: raiz, hist(max_iter)
    integer,  intent(out) :: iter, estado
    real(dp) :: x, paso, dfx
    x = x0;  raiz = x0;  iter = 0;  estado = 1;  hist = 0.0_dp
    do iter = 1, max_iter
       dfx = dF(x, n)
       if (abs(dfx) < tiny(1.0_dp)) then
          estado = 3
          print '(a)', 'Newton: derivada nula'
          return
       end if
       paso = -F(x, n)/dfx                   ! corrección de Newton
       x = x + paso
       hist(iter) = x
       if (abs(paso) < tol) then
          raiz = x;  estado = 0;  return
       end if
    end do
    iter = max_iter
    raiz = x
    print '(a)', 'Newton: no convergió en max_iter iteraciones'
  end subroutine newton

  ! Secante con dos valores iniciales; hist(k) guarda x_(k+1)
  subroutine secante(n, x0, x1, tol, raiz, iter, estado, hist)
    integer,  intent(in)  :: n
    real(dp), intent(in)  :: x0, x1, tol
    real(dp), intent(out) :: raiz, hist(max_iter)
    integer,  intent(out) :: iter, estado
    real(dp) :: xa, xb, xn, den
    xa = x0;  xb = x1;  raiz = x1;  iter = 0;  estado = 1;  hist = 0.0_dp
    do iter = 1, max_iter
       den = F(xb, n) - F(xa, n)
       if (abs(den) < tiny(1.0_dp)) then
          estado = 3
          print '(a)', 'secante: denominador nulo'
          return
       end if
       xn = xb - F(xb, n)*(xb - xa)/den
       hist(iter) = xn
       xa = xb;  xb = xn
       if (abs(xb - xa) < tol) then
          raiz = xb;  estado = 0;  return
       end if
    end do
    iter = max_iter
    raiz = xb
    print '(a)', 'secante: no convergió en max_iter iteraciones'
  end subroutine secante

end module modulo_ceros

Writing modulo_ceros.f90


In [20]:
%%writefile programa_wien.f90
! ============================================================
! Programa:  programa_wien
! Propósito: raíces x5 y x3 de x = n(1 - e^-x) con tres métodos
!            y constante de Wien b = hc/(kB x5).
! Salida:    resultado_wien.dat, iteraciones_wien.dat, constante_wien.dat
! ============================================================
program programa_wien
  use modulo_ceros
  implicit none
  real(dp), parameter :: tol = 1.0e-10_dp
  real(dp), parameter :: h_planck = 6.62607015e-34_dp
  real(dp), parameter :: c_luz    = 299792458.0_dp
  real(dp), parameter :: k_boltz  = 1.380649e-23_dp
  real(dp) :: raiz, hist(max_iter), x5, x3, b
  integer  :: u_res, u_it, u_cte, iter, estado, k, m, i, n

  open(newunit=u_res, file='resultado_wien.dat',   status='replace', action='write')
  open(newunit=u_it,  file='iteraciones_wien.dat', status='replace', action='write')

  do i = 1, 2
     n = merge(5, 3, i == 1)                 ! primero n = 5 y luego n = 3
     do m = 1, 3                             ! 1 bisección, 2 Newton, 3 secante
        select case (m)
        case (1); call biseccion(n, 1.0_dp, 10.0_dp, tol, raiz, iter, estado, hist)
        case (2); call newton(n, 10.0_dp, tol, raiz, iter, estado, hist)
        case (3); call secante(n, 10.0_dp, 9.0_dp, tol, raiz, iter, estado, hist)
        end select
        if (estado /= 0) stop 'programa_wien: un método no convergió'
        write(u_res, '(2i6,es24.16,i6)') n, m, raiz, iter
        if (n == 5) then
           if (m == 2) x5 = raiz             ! x5 por Newton
           do k = 1, iter
              write(u_it, '(2i6,es24.16)') m, k, hist(k)
           end do
        else if (m == 2) then
           x3 = raiz
        end if
        print '(a,i1,a,i1,a,f18.12,a,i0)', 'n=', n, ' método=', m, '  raíz=', raiz, '  iteraciones=', iter
     end do
  end do

  ! Newton con n = 5 desde x0 = 0.5: converge a la raíz trivial x = 0
  call newton(5, 0.5_dp, tol, raiz, iter, estado, hist)
  if (estado /= 0) stop 'programa_wien: Newton desde 0.5 no convergió'
  write(u_res, '(2i6,es24.16,i6)') 5, 4, raiz, iter
  print '(a,es12.3,a,i0)', 'Newton desde x0=0.5: raíz=', raiz, '  iteraciones=', iter
  close(u_res);  close(u_it)

  b = h_planck*c_luz/(k_boltz*x5)            ! constante de Wien [m K]
  open(newunit=u_cte, file='constante_wien.dat', status='replace', action='write')
  write(u_cte, '(3es24.16)') x5, x3, b
  close(u_cte)
  print '(/,a,f14.10,a,f14.10)', 'x5 = ', x5, '   x3 = ', x3
  print '(a,es16.9,a)', 'b  = ', b, ' m K'
end program programa_wien

Writing programa_wien.f90


In [21]:
# Compila y ejecuta la Parte B (ejecute esta celda cuando sus archivos estén listos)
!gfortran -fcheck=bounds -o programa_wien modulo_ceros.f90 programa_wien.f90 && ./programa_wien

n=5 método=1  raíz=    4.965114231796  iteraciones=37
n=5 método=2  raíz=    4.965114231744  iteraciones=4
n=5 método=3  raíz=    4.965114231744  iteraciones=5
n=3 método=1  raíz=    2.821439372106  iteraciones=37
n=3 método=2  raíz=    2.821439372122  iteraciones=5
n=3 método=3  raíz=    2.821439372122  iteraciones=6
Newton desde x0=0.5: raíz=   8.945E-17  iteraciones=6

x5 =   4.9651142317   x3 =   2.8214393721
b  =  2.897771955E-03 m K


### C. Parte Fortran: constante de Stefan-Boltzmann

**Actividades**

1. Agregue a su `modulo_romberg.f90` del Taller 7 los integrandos de esta parte, sin cambiar la integración ni su criterio de parada. Los extremos $t = 0$ y $t = 1$ deben dar el límite correcto del integrando, que es cero en ambos.
2. Escriba `programa_sigma.f90`, que calcule $I$ en $t \in [0, 1]$ con tolerancia $10^{-8}$ y obtenga $\sigma$.
3. Calcule también $\int_0^{100} x^3/(e^x - 1)\,dx$ con el mismo Romberg y la misma tolerancia, **sin** cambio de variable.
4. Compare los dos resultados con $\pi^4/15$ y explique en el reporte lo que ocurre en el paso 3.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `modulo_romberg.f90` | Módulo del Taller 7 con los integrandos nuevos |
| `programa_sigma.f90` | Usa el módulo; escribe el archivo `.dat` de esta parte |
| `resultado_sigma.dat` | Dos filas, $k_{\text{final}}$, evaluaciones, integral, $\sigma$: fila 1 con cambio de variable, fila 2 sin cambio de variable. El nivel $k = 0$ es el trapecio inicial y las evaluaciones son $2^k + 1$ |

<div style="border-left: 5px solid #2E7D32; background-color: #EAF5EA; padding: 10px 14px; margin: 8px 0;">

**Consejo.** Para $t$ cercano a 1, $x$ es tan grande que $e^x$ desborda la doble precisión a partir de $x \approx 709$. El integrando es prácticamente cero mucho antes: decida a partir de qué valor de $x$ puede devolver cero.

</div>

**Solución — Parte C.** El módulo de Romberg incluye el integrando transformado (`sel = 1`) y el directo (`sel = 2`). Si ya tiene su `modulo_romberg.f90` del Taller 7, agregue a él las funciones `planck` e `integrando` y conserve su integración.

In [22]:
%%writefile modulo_romberg.f90
! ============================================================
! Módulo:    modulo_romberg
! Propósito: integración de Romberg con el criterio de la diagonal
!            |R(k,k) - R(k-1,k-1)| < tol, e integrandos de Planck.
! Nivel k:   k = 0 es el trapecio inicial; el nivel k usa 2**k + 1 nodos.
! Integrandos (sel):
!   1: g(t) = x**3/(e**x - 1) / (1 - t)**2,  x = t/(1 - t)   [0,1]
!   2: x**3/(e**x - 1)                                        [0,inf)
! ============================================================
module modulo_romberg
  implicit none
  integer, parameter :: dp = kind(1.0d0)
  integer, parameter :: k_max = 20             ! límite de niveles
  real(dp), parameter :: x_corte = 700.0_dp    ! e**x desborda cerca de 709
contains

  ! planck: x**3/(e**x - 1) con límite 0 en x = 0 y cero para x grande
  real(dp) function planck(x)
    real(dp), intent(in) :: x
    if (x <= 0.0_dp .or. x > x_corte) then
       planck = 0.0_dp
    else
       planck = x**3/(exp(x) - 1.0_dp)
    end if
  end function planck

  ! integrando: selecciona la función a integrar
  real(dp) function integrando(sel, s)
    integer,  intent(in) :: sel
    real(dp), intent(in) :: s
    real(dp) :: x
    select case (sel)
    case (1)                                  ! s es t en [0,1]
       if (s <= 0.0_dp .or. s >= 1.0_dp) then
          integrando = 0.0_dp                 ! límites en t = 0 y t = 1
       else
          x = s/(1.0_dp - s)
          integrando = planck(x)/(1.0_dp - s)**2
       end if
    case (2)                                  ! s es x
       integrando = planck(s)
    case default
       stop 'integrando: selector inválido'
    end select
  end function integrando

  ! romberg: integral de integrando(sel, .) en [a, b]
  subroutine romberg(sel, a, b, tol, integral, k_final, n_eval)
    integer,  intent(in)  :: sel
    real(dp), intent(in)  :: a, b, tol
    real(dp), intent(out) :: integral
    integer,  intent(out) :: k_final, n_eval
    real(dp) :: R(0:k_max, 0:k_max), h, suma, factor
    integer  :: k, j, i

    h = b - a
    R(0,0) = 0.5_dp*h*(integrando(sel, a) + integrando(sel, b))   ! trapecio inicial
    integral = R(0,0);  k_final = 0
    do k = 1, k_max
       h = 0.5_dp*h
       suma = 0.0_dp
       do i = 1, 2**(k-1)                      ! nodos nuevos a + (2i-1) h
          suma = suma + integrando(sel, a + real(2*i - 1, dp)*h)
       end do
       R(k,0) = 0.5_dp*R(k-1,0) + h*suma       ! trapecio refinado
       factor = 1.0_dp
       do j = 1, k                             ! extrapolación de Richardson
          factor = 4.0_dp*factor
          R(k,j) = (factor*R(k,j-1) - R(k-1,j-1))/(factor - 1.0_dp)
       end do
       integral = R(k,k);  k_final = k
       if (abs(R(k,k) - R(k-1,k-1)) < tol) exit
    end do
    if (k_final == k_max .and. abs(R(k_max,k_max) - R(k_max-1,k_max-1)) >= tol) &
         print '(a)', 'romberg: no convergió en k_max niveles'
    n_eval = 2**k_final + 1
  end subroutine romberg

end module modulo_romberg

Writing modulo_romberg.f90


In [23]:
%%writefile programa_sigma.f90
! ============================================================
! Programa:  programa_sigma
! Propósito: constante de Stefan-Boltzmann con Romberg, con y sin
!            cambio de variable x = t/(1 - t).
! Salida:    resultado_sigma.dat, tabla en pantalla
! ============================================================
program programa_sigma
  use modulo_romberg
  implicit none
  real(dp), parameter :: pi = 4.0_dp*atan(1.0_dp)
  real(dp), parameter :: h_planck = 6.62607015e-34_dp
  real(dp), parameter :: c_luz    = 299792458.0_dp
  real(dp), parameter :: k_boltz  = 1.380649e-23_dp
  real(dp), parameter :: tol = 1.0e-8_dp
  real(dp), parameter :: i_exacta = pi**4/15.0_dp
  real(dp) :: factor, integral, sigma
  integer  :: u, fila, k_final, n_eval

  factor = 2.0_dp*pi*k_boltz**4/(h_planck**3*c_luz**2)
  open(newunit=u, file='resultado_sigma.dat', status='replace', action='write')
  print '(a)', ' caso                      k  evaluaciones        integral          sigma'
  do fila = 1, 2
     if (fila == 1) then
        call romberg(1, 0.0_dp, 1.0_dp, tol, integral, k_final, n_eval)     ! con cambio de variable
     else
        call romberg(2, 0.0_dp, 100.0_dp, tol, integral, k_final, n_eval)   ! sin cambio de variable
     end if
     sigma = factor*integral
     write(u, '(2i6,2es24.16)') k_final, n_eval, integral, sigma
     print '(a,i4,i8,es20.12,es16.8)', merge(' con cambio de variable ', ' sin cambio de variable ', fila == 1), &
           k_final, n_eval, integral, sigma
  end do
  close(u)
  print '(/,a,es20.12)', 'pi**4/15 = ', i_exacta
  print '(a,es16.8)',    'sigma exacta = ', factor*i_exacta
end program programa_sigma

Writing programa_sigma.f90


In [24]:
!gfortran -fcheck=bounds -o programa_sigma modulo_romberg.f90 programa_sigma.f90 && ./programa_sigma

 caso                      k  evaluaciones        integral          sigma
 con cambio de variable   10    1025  6.493939402267E+00  5.67037442E-08
 sin cambio de variable    1       3  1.607291539970E-15  1.40345394E-23

pi**4/15 =   6.493939402267E+00
sigma exacta =   5.67037442E-08


### D. Parte Python: máximo del espectro solar

Se modela el Sol como un cuerpo negro a $T = 5772$ K. El espectro se muestrea en una malla uniforme de longitudes de onda en $[100, 3000]$ nm, con paso $\Delta\lambda$.

**Actividades**

1. Escriba `maximo_espectro.py`. Construya $B_\lambda$ para $\Delta\lambda = 100, 50, 25$ y $12.5$ nm.
2. Para cada malla, obtenga $\lambda_{\max}$ de tres maneras: (a) mayor valor del arreglo; (b) cero del spline de la derivada central; (c) cero de la derivada del spline de $B_\lambda$. En (b) y (c), conserve los ceros donde la segunda derivada es negativa y, si queda más de uno, el más cercano al resultado de (a): en una malla gruesa el spline oscila y puede producir ceros espurios.
3. Lea $b$ de `constante_wien.dat` y obtenga el error absoluto en nm de cada resultado frente a $\lambda_{\max} = b/T$.
4. Escriba `maximo_espectro.dat` y grafique en `espectro_solar.png` el espectro de la malla de 25 nm, con $\lambda_{\max}$ marcado. Agregue en la misma figura el $\lambda$ que corresponde al máximo en frecuencia, $\lambda = hc/(k_B T x_3)$.

**Qué preparar**

| Archivo | Contenido |
|---|---|
| `maximo_espectro.py` | Construye los datos, halla los máximos, escribe los archivos y la figura |
| `maximo_espectro.dat` | Cuatro filas, una por $\Delta\lambda$: $\Delta\lambda$ [nm], error (a), error (b), error (c) [nm] |
| `espectro_solar.png` | Espectro con $\lambda_{\max}$ y el $\lambda$ del máximo en frecuencia |

**Solución — Parte D.**

In [25]:
%%writefile maximo_espectro.py
"""Máximo del espectro solar (T = 5772 K): malla, derivada central y spline.

Entrada : constante_wien.dat  (x5, x3, b)
Salida  : maximo_espectro.dat, espectro_solar.png
"""
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline

H, C, KB = 6.62607015e-34, 299792458.0, 1.380649e-23
T = 5772.0
AZUL, DORADO, ROJO, VERDE, GRIS = "#162A54", "#C9972C", "#B02A20", "#2E7D32", "#7F7F7F"


def planck(lam_nm):
    """B_lambda en W m^-2 sr^-1 m^-1 para lam en nm."""
    lam = lam_nm*1e-9
    return 2*H*C**2/lam**5/np.expm1(H*C/(lam*KB*T))


def derivada_central(y, h):
    """Derivada central en los puntos interiores (programada, sin np.gradient)."""
    return (y[2:] - y[:-2])/(2*h)


def cero_maximo(s, cercano_a):
    """Ceros del spline s con segunda derivada negativa; el más cercano a 'cercano_a'."""
    r = s.roots(extrapolate=False)
    r = r[np.isfinite(r)]
    r = r[s(r, 1) < 0]                      # solo máximos (descarta mínimos y ceros espurios)
    if len(r) == 0:
        raise ValueError("no se encontró candidato a máximo")
    return r[np.argmin(np.abs(r - cercano_a))]


x5, x3, b = np.loadtxt("constante_wien.dat")
lam_wien = b/T*1e9                          # lambda_max = b/T [nm]
lam_frec = H*C/(KB*T*x3)*1e9                # lambda del máximo en frecuencia [nm]
print(f"b = {b:.9e} m K   lambda_max (Wien) = {lam_wien:.4f} nm   lambda (max. en frecuencia) = {lam_frec:.2f} nm")

filas = []
for dl in (100.0, 50.0, 25.0, 12.5):
    lam = np.arange(100.0, 3000.0 + dl/2, dl)
    B = planck(lam)
    B = B/B.max()                           # escala 0..1: no cambia la posición de los ceros
    la = lam[np.argmax(B)]                                              # (a) mayor valor
    lb = cero_maximo(CubicSpline(lam[1:-1], derivada_central(B, dl)), la)   # (b) spline de la derivada central
    lc = cero_maximo(CubicSpline(lam, B).derivative(), la)              # (c) derivada del spline de B
    e = [abs(la - lam_wien), abs(lb - lam_wien), abs(lc - lam_wien)]
    filas.append([dl, *e])
    print(f"dl = {dl:6.1f} nm   (a) {la:9.3f}   (b) {lb:9.3f}   (c) {lc:9.3f}   "
          f"errores [nm]: {e[0]:8.3f} {e[1]:8.3f} {e[2]:8.4f}")

filas = np.array(filas)
np.savetxt("maximo_espectro.dat", filas, fmt="%.16e")
print("\nrazón entre errores consecutivos (b, c):")
print(np.round(filas[:-1, 2:]/filas[1:, 2:], 2))

# ---- figura: malla de 25 nm
dl = 25.0
lam = np.arange(100.0, 3000.0 + dl/2, dl)
B = planck(lam)
lf = np.linspace(100, 3000, 1500)
fig, ax = plt.subplots(figsize=(8, 4.4))
ax.axvspan(380, 750, color=DORADO, alpha=0.12, label="visible (380-750 nm)")
ax.plot(lf, CubicSpline(lam, B)(lf)*1e-12, color=GRIS, lw=1, alpha=0.7)
ax.plot(lam, B*1e-12, "o", ms=3, color=AZUL, label="$B_\\lambda$, malla de 25 nm")
ax.axvline(lam_wien, color=ROJO, ls="--", label=f"$\\lambda_{{\\max}} = b/T$ = {lam_wien:.1f} nm")
ax.axvline(lam_frec, color=VERDE, ls=":", lw=2, label=f"máximo en frecuencia: {lam_frec:.0f} nm")
ax.set_xlabel("longitud de onda $\\lambda$ [nm]")
ax.set_ylabel("$B_\\lambda$ [$10^{12}$ W m$^{-2}$ sr$^{-1}$ m$^{-1}$]")
ax.set_title("Espectro del Sol como cuerpo negro, $T = 5772$ K")
ax.legend(); fig.tight_layout(); fig.savefig("espectro_solar.png", dpi=150)
print("figura: espectro_solar.png")

Writing maximo_espectro.py


In [26]:
# Ejecuta la Parte D (ejecute esta celda cuando su archivo esté listo)
!python maximo_espectro.py

b = 2.897771955e-03 m K   lambda_max (Wien) = 502.0395 nm   lambda (max. en frecuencia) = 883.48 nm
dl =  100.0 nm   (a)   500.000   (b)   514.942   (c)   500.837   errores [nm]:    2.039   12.902   1.2021
dl =   50.0 nm   (a)   500.000   (b)   505.433   (c)   502.072   errores [nm]:    2.039    3.393   0.0322
dl =   25.0 nm   (a)   500.000   (b)   502.891   (c)   502.042   errores [nm]:    2.039    0.852   0.0029
dl =   12.5 nm   (a)   500.000   (b)   502.253   (c)   502.040   errores [nm]:    2.039    0.213   0.0003

razón entre errores consecutivos (b, c):
[[ 3.8  37.38]
 [ 3.98 11.07]
 [ 4.    9.29]]
figura: espectro_solar.png


### E. Preguntas de interpretación y reporte

Responda en celdas Markdown al final de este notebook, con los números de sus archivos:

1. **Costo de los métodos.** Compare las iteraciones de los tres métodos con $n = 5$. Con `iteraciones_wien.dat`, estime el orden de Newton y de la secante.
2. **Valor inicial.** ¿A qué valor llega Newton desde $x_0 = 0.5$? ¿Por qué es una raíz correcta de la ecuación, pero no la respuesta al problema físico?
3. **Parada prematura.** En la Parte C, ¿qué valor da Romberg en $[0, 100]$ y en qué nivel se detiene? Explique, con los puntos que evalúa en sus primeros niveles, por qué el criterio de la diagonal se cumple sin que la integral haya convergido. Proponga una comprobación adicional.
4. **Eslabón más débil.** En la Parte D, ¿con qué razón bajan los errores de (b) y de (c) al reducir $\Delta\lambda$ a la mitad? ¿Qué limita la precisión de (b)? ¿Por qué el error de (a) no baja en estas mallas?
5. **Dos máximos.** ¿Por qué el máximo en longitud de onda y el máximo en frecuencia no corresponden al mismo color? ¿Cuál de los dos es "el" color del Sol?

**Reporte (una página, plantilla del curso).** Las secciones de contexto físico, método y resultado se refieren a la constante de Wien y al máximo del espectro (Partes B y D). La Parte C puede alimentar la sección de obstáculo o decisión de implementación. El reporte es una síntesis y no repite las cinco respuestas.

### F. Verificación

La celda siguiente **no debe modificarse**. Lee los archivos que escribieron sus programas, recalcula de forma independiente los valores de referencia, y comprueba la forma de cada archivo. También avisa si un archivo parece anterior a su código. El taller exige los cinco PASS.

In [27]:
# ==================== VERIFICACIÓN DEL TALLER 8 Y 9 (no modificar) ====================
import os
import numpy as np
from scipy.interpolate import CubicSpline as _CS

H_, C_, KB_ = 6.62607015e-34, 299792458.0, 1.380649e-23

def raiz_referencia(n):
    x = float(n)
    for _ in range(200):
        x = n*(1 - np.exp(-x))
    return x

X5, X3 = raiz_referencia(5), raiz_referencia(3)
B_REF = H_*C_/(KB_*X5)
I_REF = np.pi**4/15
S_REF = 2*np.pi*KB_**4/(H_**3*C_**2)*I_REF
ITER_ESPERADAS = {(5,1):37, (5,2):4, (5,3):5,
                  (3,1):37, (3,2):5, (3,3):6}

_avisados = set()
def tabla(dat, fuentes, forma=None):
    for ruta in [dat, *fuentes]:
        if not os.path.isfile(ruta):
            raise ValueError(f"no existe {ruta}; ejecute desde la carpeta del taller")
    antiguos = [f for f in fuentes if os.path.getmtime(dat) < os.path.getmtime(f)]
    if antiguos and dat not in _avisados:
        print(f"[AVISO] {dat} parece anterior a su fuente; vuelva a generar si editó el código.")
        _avisados.add(dat)
    t = np.loadtxt(dat, ndmin=2)
    if forma is not None and t.shape != forma:
        raise ValueError(f"{dat}: forma {t.shape}; se espera {forma}")
    if not np.all(np.isfinite(t)):
        raise ValueError(f"{dat}: contiene NaN o infinitos")
    return t

def enteros(a, nombre):
    if not np.all(a == np.floor(a)):
        raise ValueError(f"{nombre}: se requieren valores enteros")

def resultados_wien():
    t = tabla('resultado_wien.dat', ['modulo_ceros.f90', 'programa_wien.f90'], (7,4))
    enteros(t[:,[0,1,3]], 'n, método e iteraciones')
    claves = [tuple(map(int, fila[:2])) for fila in t[:6]]
    if len(set(claves)) != 6 or set(claves) != set(ITER_ESPERADAS):
        raise ValueError('debe aparecer una vez cada combinación de n=5,3 y método=1,2,3')
    if tuple(t[6,:2]) != (5,4):
        raise ValueError('la fila 7 debe identificar n=5, método=4')
    if np.any((t[:,3] < 1) | (t[:,3] > 100)):
        raise ValueError('conteos fuera de 1 a 100')
    return t

def constantes():
    t = tabla('constante_wien.dat', ['modulo_ceros.f90', 'programa_wien.f90'], (1,3))[0]
    if abs(t[0]-X5)>2e-10 or abs(t[1]-X3)>2e-10:
        raise ValueError('x5 o x3 incorrectos en constante_wien.dat')
    if abs(t[2]-B_REF)/B_REF > 1e-9:
        raise ValueError('constante de Wien incorrecta')
    return t

resultados = []
def caso(nombre, funcion):
    try:
        detalle = funcion()
        ok = True
    except Exception as e:
        ok, detalle = False, str(e)
    resultados.append(ok)
    print(f"[{'PASS' if ok else 'FAIL'}] {nombre}" + (f" -> {detalle}" if detalle else ''))

def c1():
    t = resultados_wien()
    for n, m, raiz, it in t[:6]:
        ref = X5 if n == 5 else X3
        if abs(raiz-ref)>2e-10:
            raise ValueError(f'n={n:g}, método={m:g}: raíz incorrecta')
        esperado = ITER_ESPERADAS[int(n),int(m)]
        if it != esperado:
            raise ValueError(f'n={n:g}, método={m:g}: {it:g} iteraciones; se esperan {esperado}')
    return f'x5={X5:.10f}, x3={X3:.10f}; conteos correctos'

def historia_referencia(m):
    # Recurrencias independientes para verificar los registros de n=5.
    f = lambda x: x-5*(1-np.exp(-x))
    df = lambda x: 1-5*np.exp(-x)
    a,b,prev = 1.0,10.0,10.0
    x = {1:0., 2:10., 3:9.}[m]
    valores=[]
    for _ in range(ITER_ESPERADAS[5,m]):
        if m==1:
            z=(a+b)/2
            if f(a)*f(z)<=0: b=z
            else: a=z
        elif m==2: z=x-f(x)/df(x)
        else: z=x-f(x)*(x-prev)/(f(x)-f(prev))
        valores.append(z)
        prev,x=x,z
    return np.array(valores)

def c2():
    t = tabla('iteraciones_wien.dat', ['modulo_ceros.f90','programa_wien.f90'])
    if t.shape[1]!=3: raise ValueError('iteraciones_wien.dat debe tener tres columnas')
    enteros(t[:,:2], 'método y contador')
    if set(t[:,0])!={1,2,3}: raise ValueError('faltan métodos o hay identificadores inválidos')
    for m in range(1,4):
        filas=t[t[:,0]==m]
        ref=historia_referencia(m)
        if len(filas)!=len(ref): raise ValueError(f'método {m}: número de registros incorrecto')
        if not np.array_equal(filas[:,1], np.arange(1,len(ref)+1)):
            raise ValueError(f'método {m}: use contadores consecutivos desde 1, en orden')
        if not np.allclose(filas[:,2],ref,atol=2e-13,rtol=2e-14):
            raise ValueError(f'método {m}: trayectoria incorrecta o pocas cifras en el archivo')
    return 'tres historiales completos'

def c3():
    x5,x3,b=constantes()
    t=resultados_wien()
    if abs(t[6,2])>1e-8: raise ValueError('Newton desde 0.5 debe llegar a la raíz trivial')
    return f'b={b:.9e} m K; raíz trivial comprobada'

def c4():
    t=tabla('resultado_sigma.dat', ['modulo_romberg.f90','programa_sigma.f90'], (2,4))
    enteros(t[:,:2], 'nivel y evaluaciones de Romberg')
    for k,ne,I,s in t:
        if k<0 or k>50 or ne<2:
            raise ValueError('nivel o conteo de Romberg inválido')
        if int(ne)!=2**int(k)+1:
            raise ValueError('use nivel k=0 para el trapecio inicial y cuente nodos únicos: 2**k+1')
        factor=2*np.pi*KB_**4/(H_**3*C_**2)
        if not np.isclose(s,factor*I,rtol=1e-8,atol=1e-30):
            raise ValueError('sigma no corresponde a la integral de su fila')
    if abs(t[0,2]-I_REF)>1e-8 or abs(t[0,3]-S_REF)/S_REF>1e-8:
        raise ValueError('la integral transformada o sigma no alcanzan la precisión solicitada')
    prematura=100/6*(4*50**3/np.expm1(50)+100**3/np.expm1(100))
    if t[1,0]!=1 or not np.isclose(t[1,2],prematura,rtol=1e-6,atol=1e-18):
        raise ValueError('fila 2: debe mostrar la parada prematura en k = 1')
    return f'sigma={t[0,3]:.9e} W/m²/K⁴; parada prematura reproducida'

def c5():
    constantes()
    t=tabla('maximo_espectro.dat', ['maximo_espectro.py','constante_wien.dat'], (4,4))
    if not np.array_equal(t[:,0],[100.,50.,25.,12.5]):
        raise ValueError('use, en este orden, pasos de 100, 50, 25 y 12.5 nm')
    if np.any(t[:,1:]<0): raise ValueError('los errores absolutos no pueden ser negativos')
    ref=B_REF/5772*1e9
    for fila in t:
        dl=fila[0]
        l=np.arange(100.,3000.+dl/2,dl)
        B=1/l**5/np.expm1(H_*C_/(l*1e-9*KB_*5772))
        la=l[np.argmax(B)]
        def cero(s):
            r=s.roots(extrapolate=False)
            r=r[np.isfinite(r)]
            r=r[s(r,1)<0]
            if len(r)==0: raise ValueError('no se encontró candidato a máximo')
            return r[np.argmin(abs(r-la))]
        esperados=[abs(la-ref),
            abs(cero(_CS(l[1:-1],(B[2:]-B[:-2])/(2*dl)))-ref),
            abs(cero(_CS(l,B).derivative())-ref)]
        for nombre,val,esp in zip(['a','b','c'],fila[1:],esperados):
            if abs(val-esp)>1e-4+0.01*esp:
                raise ValueError(f'paso {dl:g} nm, método {nombre}: error incompatible')
    if not os.path.isfile('espectro_solar.png'):
        raise ValueError('falta espectro_solar.png')
    return 'cuatro mallas correctas'

caso('1. Raíces e iteraciones de los tres métodos', c1)
caso('2. Historiales de iteración y convergencia', c2)
caso('3. Constante de Wien y Newton desde x0=0.5', c3)
caso('4. Stefan–Boltzmann y parada prematura', c4)
caso('5. Máximo del espectro solar en cuatro mallas', c5)
print(f'\nResultado: {sum(resultados)} de {len(resultados)} PASS')

[PASS] 1. Raíces e iteraciones de los tres métodos -> x5=4.9651142317, x3=2.8214393721; conteos correctos
[PASS] 2. Historiales de iteración y convergencia -> tres historiales completos
[PASS] 3. Constante de Wien y Newton desde x0=0.5 -> b=2.897771955e-03 m K; raíz trivial comprobada
[PASS] 4. Stefan–Boltzmann y parada prematura -> sigma=5.670374419e-08 W/m²/K⁴; parada prematura reproducida
[PASS] 5. Máximo del espectro solar en cuatro mallas -> cuatro mallas correctas

Resultado: 5 de 5 PASS


## E. Respuestas de interpretación

### Cálculo de apoyo: orden de convergencia (E1)

In [28]:
# Orden de Newton y de la secante a partir de iteraciones_wien.dat (n = 5)
t = np.loadtxt("iteraciones_wien.dat")
x5_ref = 5.0
for _ in range(300):                           # raíz de referencia por iteración de punto fijo
    x5_ref = 5*(1 - np.exp(-x5_ref))
for m, nombre, x_ini in [(2, "Newton", 10.0), (3, "secante", 9.0)]:
    x = t[t[:, 0] == m][:, 2]
    e = np.abs(np.concatenate([[x_ini], x]) - x5_ref)   # incluye el error del valor inicial
    e = e[e > 1e-13]                                    # descarta errores dominados por el redondeo
    print(nombre, "errores:", np.array2string(e, precision=2))
    print(nombre, "p (terna a terna):", np.round(np.log(e[2:]/e[1:-1])/np.log(e[1:-1]/e[:-2]), 2))

Newton errores: [5.03e+00 3.35e-02 1.98e-05 7.11e-12]
Newton p (terna a terna): [1.48 2.  ]
secante errores: [4.03e+00 3.27e-02 8.59e-04 5.02e-07 7.78e-12]
secante p (terna a terna): [0.76 2.05 1.49]


**1. Costo de los métodos.** Con $n=5$ las iteraciones son 37 (bisección), 4 (Newton) y 5 (secante); con $n=3$, 37, 5 y 6. La bisección coincide con la predicción $n \ge \log_2(9/10^{-10}) = 36.4 \to 37$. En evaluaciones de función Newton usa 2 por iteración (F y F′), unas 8 en total; la secante usa 1 por iteración más 2 iniciales, unas 7; la bisección, unas 37. Los errores de Newton son $3.4\times10^{-2}$, $2.0\times10^{-5}$ y $7.1\times10^{-12}$; el cociente $e_{n+1}/e_n^2 \approx 1.8\times10^{-2}$ es casi constante e igual a $F''/(2F') = 0.018$ en $x_5$, y la última terna da $p \approx 2.0$. En la secante la última terna libre de redondeo da $p \approx 1.5$, cerca del valor teórico 1.618; con solo cuatro errores utilizables la estimación es ruidosa.

**2. Valor inicial.** Newton desde $x_0 = 0.5$ llega a $x \approx 9\times10^{-17}$, es decir, a la **raíz trivial** $x=0$, en 6 iteraciones. Es una raíz correcta de $x = 5(1-e^{-x})$, pero $x = hc/(\lambda k_B T) = 0$ corresponde a $\lambda \to \infty$, donde el espectro tiende a cero: no es el máximo. La razón es la pendiente: $F'(x) = 1 - 5e^{-x}$ se anula en $x = \ln 5 \approx 1.61$; en $x_0 = 0.5$, $F' = -2.03$ y $F = -1.47$, la tangente cruza el eje en $x_1 \approx -0.22$ y el método cae en $x=0$. Desde $x_0 = 10$ la pendiente es casi 1 y el método camina hacia $x_5$. La ecuación admite dos soluciones y solo una es la respuesta física.

**3. Parada prematura.** Romberg en $[0,100]$ devuelve $1.6\times10^{-15}$ y se detiene en $k=1$ con 3 evaluaciones, cuando la integral vale $6.4939$. Sus primeros niveles evalúan solo $x = 0, 100$ ($k=0$) y $x=50$ ($k=1$), puntos donde el integrando es $0$, $3.7\times10^{-38}$ y $2.6\times10^{-17}$, mientras que toda la masa está en $x \sim 1$–$10$ (el máximo está en $x \approx 2.8$). Las dos primeras estimaciones de la diagonal, $R(0,0)\approx2\times10^{-36}$ y $R(1,1)\approx1.6\times10^{-15}$, difieren en menos de $10^{-8}$ y el criterio se cumple sin que la integral haya convergido. Comprobaciones adicionales: exigir un nivel mínimo (por ejemplo $k \ge 4$ o 5), usar una tolerancia **relativa** al valor de la integral, comparar con el resultado de otra partición o de otro límite superior (por ejemplo $[0,50]$ frente a $[0,100]$), o aplicar el cambio de variable $x = t/(1-t)$, que concentra la parte importante del integrando en puntos que sí se evalúan.

**4. Eslabón más débil.** Al reducir $\Delta\lambda$ a la mitad, el error de (b) baja con razón $3.8$, $3.98$ y $4.0$: orden 2. El de (c) baja con razón $37$, $11$ y $9.3$, que tiende hacia $\approx 8$ (orden 3, consistente con que la derivada de un spline cúbico pierde un orden frente a la función). La precisión de (b) la limita la **derivada central**, de orden 2: el spline interpola bien un dato que ya es impreciso. El error de (a) se queda en $2.039$ nm en las cuatro mallas porque el máximo real, $502.04$ nm, está entre nodos y el mayor valor de un arreglo solo puede ser un nodo: el nodo $500$ nm existe en todas las mallas y es el más cercano. Solo bajaría con pasos menores que unos 4 nm, y de manera irregular.

**5. Dos máximos.** $B_\nu\,d\nu = B_\lambda\,d\lambda$ y $|d\lambda/d\nu| = \lambda^2/c$, así que $B_\nu = B_\lambda\lambda^2/c$: el factor $\lambda^2$ desplaza el máximo hacia longitudes de onda mayores. Por eso $\lambda_{\max} = b/T = 502.0$ nm (en el verde) y el máximo en frecuencia está en $\lambda = hc/(k_B T x_3) = 883.5$ nm (infrarrojo), con cociente $x_5/x_3 = 1.76$. Son los máximos de dos funciones distintas, densidad por unidad de longitud de onda y densidad por unidad de frecuencia; ninguno es más correcto. Decir "el" color del Sol exige elegir una variable (y, para el color percibido, considerar la respuesta del ojo y la atmósfera); el espectro es ancho en el visible y la luz solar se ve blanca.

## Lo que se aprende de este cuaderno

- **Localizar antes de refinar:** el teorema del valor intermedio y un barrido de cambios de signo dan los intervalos de arranque; no detecta raíces dobles ni pares de raíces más cercanas que el paso.
- **Bisección:** siempre converge, orden 1, el número de iteraciones se conoce de antemano ($n \ge \log_2[(b-a)/\varepsilon]$); el error real oscila, la cota no.
- **Newton:** orden 2 ($e_{n+1} \approx C e_n^2$), costo de dos evaluaciones por paso y sin garantía global: depende del valor inicial, de que $f' \neq 0$ y puede llevar a otra raíz (aquí la trivial).
- **Secante:** sin derivada, orden $\approx 1.618$, una evaluación nueva por paso.
- **Orden de convergencia medido:** $p \approx \ln(e_{n+1}/e_n)/\ln(e_n/e_{n-1})$, con errores por encima del redondeo.
- **Criterios de parada:** intervalo, paso, residuo y máximo de iteraciones; el residuo solo no basta y siempre se revisa al final.
- **Extremos:** ceros de $f'$ clasificados con el signo de $f''$; mayor valor del arreglo (error irregular), cero lineal, spline de la derivada (limitado por la derivada central) y derivada del spline de $f$ (la mejor).
- **Integración:** una parada por diagonal puede engañar si los primeros nodos no ven la función; el cambio de variable $x = t/(1-t)$ evita la parada prematura, y hay que decidir desde qué $x$ el integrando se puede tomar como cero para evitar el desbordamiento de $e^x$.
- **Física:** la ley de Wien sale de $x = n(1 - e^{-x})$ ($n=5$ en longitud de onda, $n=3$ en frecuencia) y la de Stefan-Boltzmann de $I = \pi^4/15$.